# Query-adapter pilot: preparation (twelve carriers), Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16, greedy, thinking off. **No adapter is trained here.** Frozen construction: generic support from the timing12 templates, result selection, positions, prompts, 384 plan / 48 answer tokens.

Splits (already written, hashes registered in `configs/pinned.json` under `study_plan4_qadapt`):
- **train**: 96 base sheets (32 per level)
- **dev**: 12 base sheets + 12 A-change pairs
- **eval**: 24 base sheets + 24 A-change pairs; **no A sheets are generated for eval here**

Phases:
- **Cell 4**: A's ordinary written pass for train + dev (120 sheets, about 1 minute each, resumes)
- **Cell 5**: construction check: templates must equal the frozen timing12 templates; every sheet must build tk_tv and fk_fv with unchanged support, IDs, positions and masks; support identical within each template
- **Cell 6**: teacher targets on train base sheets: fk_fv plan + answer (unfiltered) and ordinary tk_tv (the gap before training)

The nine timing12 cases are used only to confirm the templates match; they are not trained on.

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`, `data/plan4_qadapt_train.jsonl`, `data/plan4_qadapt_dev.jsonl`, `data/plan4_qadapt_eval.jsonl`, and the folder `results_stage_plan4_refval12/` (for the frozen timing12 A artifacts). If A sheets were already written in an earlier session, also upload the downloaded `results_stage_plan4_qadapt_prep/` folder at the repo root; the stage restores from it and Cell 4 skips finished sheets.

Download `/kaggle/working/results_stage_plan4_qadapt_prep`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_qadapt_prep.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

NEED = ("plan4_timing12.jsonl", "plan4_qadapt_train.jsonl", "plan4_qadapt_dev.jsonl", "plan4_qadapt_eval.jsonl")
REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_qadapt_prep.py").is_file() and all((cand / "data" / n).is_file() for n in NEED):
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError(f"Need src/stage_plan4_qadapt_prep.py and data/{NEED}.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_qadapt_prep.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
print("timing12 A artifacts", (REPO / "results_stage_plan4_refval12" / "a_artifacts_12.jsonl").is_file())
for base in (WORK / "results_stage_plan4_qadapt_prep", REPO / "results_stage_plan4_qadapt_prep"):
    for name in ("a_artifacts_train.jsonl", "a_artifacts_dev.jsonl"):
        p = base / name
        print("saved A sheets", p, sum(1 for l in p.read_text(encoding="utf-8").splitlines() if l.strip()) if p.is_file() else "missing")

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_qadapt_prep", "--phase", "data"])
print("data exit", rc)
if rc != 0:
    raise RuntimeError("Split files do not match the registered hashes. Do not regenerate them.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_qadapt_prep", "--phase", "write"])
print("write exit", rc, "(rerun this cell to resume if the session stops)")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_qadapt_prep", "--phase", "check"])
print("check exit", rc)
src = WORK / "results_stage_plan4_qadapt_prep"
if not (src / "check_summary.json").is_file():
    src = REPO / "results" / "stage_plan4_qadapt_prep" / "run"
check = json.loads((src / "check_summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: v for k, v in check.items() if k not in ("rows", "support_within_template")}, indent=1))
print("support within template (ids required; tensor differences reported)")
for t, b in check["support_within_template"].items():
    print(" ", t, b)
bad = [r["a_state_key"] for r in check["rows"] if not r["ok"]]
print("failing sheets", bad[:10])
if not check["all_passed"]:
    raise RuntimeError("Construction check failed. Do not run the teacher phase.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_qadapt_prep", "--phase", "teacher"])
print("teacher exit", rc, "(rerun to resume)")
data = json.loads((src / "teacher_summary.json").read_text(encoding="utf-8"))
for cond in ("tk_tv", "fk_fv"):
    print(cond, "labels", data[cond]["labels"])
    print(cond, "final choice", data[cond]["final_choice"])
print("paired", data["paired"])
print("teacher targets", data["teacher_targets"])